# DS23 · Session 42 · Agent Fundamentals 1/2
## Building an agent from scratch: the loop, tools, and function calling

**BIU DS23 by Shlomit Levavi**

This notebook builds a complete agent in about 60 lines of plain Python, with no framework.
We start with a single `calculator` tool, then promote our Module 5 RAG pipeline into a
`search_regulation` tool. Everything runs through one `chat()` wrapper and one budget guard.

**The punchline of the whole module:** *Autonomy is a budget, not a switch.*


### Setup: backend, secrets, and the budget guard
One configuration cell for the whole module. The budget guard is locked at the top of every agent notebook.

In [1]:
import os, json, re, math

SEED = 42

# ---- backend config (Module 5 architecture, now tool-aware) ----
BACKEND = "groq"            # tool calling needs an instruction-following model
MODEL   = "llama-3.3-70b-versatile"

# ---- the budget guard: a hard law, not a suggestion ----
MAX_ITERATIONS = 8          # hard stop for every agent loop, no exceptions
COST_CAP_USD   = 0.10       # per task; the loop exits gracefully on BudgetExceeded

# Groq public pricing for this model (teaching-representative), USD per token
PRICE_IN  = 0.59 / 1_000_000
PRICE_OUT = 0.79 / 1_000_000

class BudgetExceeded(Exception):
    pass

print("Backend:", BACKEND, "| model:", MODEL)
print("Guard  : MAX_ITERATIONS =", MAX_ITERATIONS, "| COST_CAP_USD =", COST_CAP_USD)

Backend: groq | model: llama-3.3-70b-versatile
Guard  : MAX_ITERATIONS = 8 | COST_CAP_USD = 0.1


### The `chat()` wrapper
In class this calls Groq. So the notebook is fully reproducible and never depends on a live API,
we ship a deterministic offline stub that plays back a saved trace. Set `USE_LIVE = True` in class
once keys are verified. The agent loop below does not know or care which one answers.

In [2]:
USE_LIVE = False   # in class, flip to True after verifying the Groq key in Colab Secrets

# --- a tiny token estimator, good enough for the cost lesson (not a real tokenizer) ---
def estimate_tokens(text):
    # ~4 chars per token is the standard rough rule for English
    return max(1, math.ceil(len(text) / 4))

# --- LIVE backend (used in class) ---
def _chat_live(messages, tools=None, temperature=0):
    from groq import Groq
    from google.colab import userdata
    client = Groq(api_key=userdata.get("GROQ_API_KEY"))
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools,
        temperature=temperature, tool_choice="auto",
    )
    return resp

# --- OFFLINE stub: a scripted, deterministic controller for reproducibility ---
class _Msg:
    def __init__(self, content=None, tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls

class _ToolCall:
    def __init__(self, name, args):
        self.id = "call_" + name
        self.type = "function"
        self.function = type("F", (), {"name": name, "arguments": json.dumps(args)})

def _stub_decide(messages):
    """Deterministic policy that mimics a tool-using model for the demo tasks."""
    last = messages[-1]
    text = " ".join(str(m.get("content", "")) for m in messages).lower()
    # if the most recent message is a tool result, decide next step from what we have
    have_days = any(m.get("role") == "tool" and "72" in str(m.get("content","")) for m in messages)
    have_date = any(m.get("role") == "tool" and "2026-" in str(m.get("content","")) for m in messages)
    asked_math = "compute" in text or "what is" in text or "+" in text or "-" in text
    asked_reg  = "gdpr" in text or "regulation" in text or "notify" in text or "breach" in text
    asked_date = "deadline" in text or "date" in text or "days" in text

    # calculator-only task
    if asked_math and not asked_reg:
        if last.get("role") == "tool":
            return _Msg(content="The result is " + str(last["content"]).strip() + ".")
        expr = "90 - 17"
        m = re.search(r"(\d+)\s*([+\-*/])\s*(\d+)", text)
        if m: expr = f"{m.group(1)} {m.group(2)} {m.group(3)}"
        return _Msg(tool_calls=[_ToolCall("calculator", {"expression": expr})])

    # regulation + date task
    if asked_reg or asked_date:
        if not have_days:
            return _Msg(tool_calls=[_ToolCall("search_regulation",
                        {"query": "breach notification deadline", "regulation": "GDPR"})])
        if have_days and not have_date:
            return _Msg(tool_calls=[_ToolCall("date_calculator",
                        {"start": "2026-10-01", "days": 3})])
        return _Msg(content=("Under GDPR Article 33 you must notify within 72 hours; "
                             "starting 2026-10-01 the deadline is 2026-10-04."))
    # fallback
    return _Msg(content="I do not have a tool for that.")

def chat(messages, tools=None, temperature=0):
    """Single wrapper. Returns (message, usage_in_tokens, usage_out_tokens)."""
    if USE_LIVE:
        resp = _chat_live(messages, tools, temperature)
        m = resp.choices[0].message
        u = resp.usage
        msg = _Msg(content=m.content,
                   tool_calls=getattr(m, "tool_calls", None))
        return msg, u.prompt_tokens, u.completion_tokens
    # offline
    msg = _stub_decide(messages)
    in_tok  = estimate_tokens(json.dumps(messages))
    out_tok = estimate_tokens(msg.content or json.dumps(
              [{"name": tc.function.name, "arguments": tc.function.arguments}
               for tc in (msg.tool_calls or [])]))
    return msg, in_tok, out_tok

print("chat() ready. USE_LIVE =", USE_LIVE)

chat() ready. USE_LIVE = False


## Part A · From thinking to acting

### Tool 1: `calculator`
A tool is just a function with a docstring and an argument schema. Nothing more mysterious than that.
The model is genuinely bad at arithmetic, so this tool lets it do what it cannot do alone.

In [3]:
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression. Use for any exact calculation."""
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "error: unsupported characters in expression"
    try:
        value = eval(expression, {"__builtins__": {}}, {})
    except Exception as e:
        return f"error: {e}"
    return str(value)

# the schema the MODEL sees (this is a prompt, not just documentation)
CALCULATOR_SCHEMA = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate an exact arithmetic expression like '90 - 17'. "
                       "Use whenever the task needs a precise number.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
}

# quick self-check
print(calculator("90 - 17"))
print(calculator("(3 + 4) * 2"))

73
14


### The agent loop
The entire agent is one `while` loop. No magic, just flow control with three stop conditions.
We track a full trace and the running cost as we go.

In [4]:
# registry mapping tool name -> (python function, schema)
TOOLS = {
    "calculator": (calculator, CALCULATOR_SCHEMA),
}

def run_tool(name, args):
    fn, _ = TOOLS[name]
    return fn(**args)

def run_agent(task, tool_names, verbose=True):
    """Run the agent loop until a final answer or the budget guard fires."""
    schemas  = [TOOLS[n][1] for n in tool_names]
    messages = [{"role": "user", "content": task}]
    trace, iterations = [], 0
    total_in = total_out = 0
    spent = 0.0

    while True:
        iterations += 1
        msg, in_tok, out_tok = chat(messages, tools=schemas)
        total_in += in_tok; total_out += out_tok
        spent = total_in * PRICE_IN + total_out * PRICE_OUT

        if getattr(msg, "tool_calls", None):
            tc = msg.tool_calls[0]
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            # validate arguments against the schema keys (Assert v3)
            required = TOOLS[name][1]["function"]["parameters"].get("required", [])
            assert all(k in args for k in required), f"missing args for {name}"
            result = run_tool(name, args)
            trace.append({"iter": iterations, "tool": name, "args": args, "result": result})
            if verbose:
                print(f"[iter {iterations}] tool_call: {name}({args})")
                print(f"          result:    {result}")
            messages.append({"role": "assistant", "content": None,
                             "tool_calls": [{"id": tc.id, "type": "function",
                                             "function": {"name": name,
                                                          "arguments": tc.function.arguments}}]})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            trace.append({"iter": iterations, "final": msg.content})
            if verbose:
                print(f"[iter {iterations}] final:     {msg.content}")
            return {"answer": msg.content, "trace": trace, "iterations": iterations,
                    "tokens_in": total_in, "tokens_out": total_out, "cost_usd": spent}

        # ---- the budget guard ----
        if iterations >= MAX_ITERATIONS or spent >= COST_CAP_USD:
            return {"answer": "I could not finish within the allotted budget.",
                    "trace": trace, "iterations": iterations,
                    "tokens_in": total_in, "tokens_out": total_out, "cost_usd": spent}

print("Agent loop defined.")

Agent loop defined.


### Demo A: the model alone vs. the agent
The model cannot do the arithmetic. The agent calls the tool and gets it right.

In [5]:
result = run_agent("What is 90 minus 17? Use a tool for the exact number.",
                   tool_names=["calculator"])
print("\nANSWER:", result["answer"])
print("iterations:", result["iterations"], "| cost: $%.6f" % result["cost_usd"])

# Assert v3.1: assert on the trajectory, not the generated text
assert result["iterations"] <= MAX_ITERATIONS
assert any(step.get("tool") == "calculator" for step in result["trace"])
print("trajectory asserts passed.")

[iter 1] tool_call: calculator({'expression': '90 - 17'})
          result:    73
[iter 2] final:     The result is 73.

ANSWER: The result is 73.
iterations: 2 | cost: $0.000082
trajectory asserts passed.


## Part B · Our tools get promoted

### Tool 2: `search_regulation` (the Module 5 RAG, now a tool)
Your Module 5 regulation assistant was a RAG pipeline. Here it is wrapped as one function the
agent calls by itself. Offline, we ship a tiny fixed index so the notebook is reproducible; in
class this is the real Chroma index from S40.

In [6]:
# a tiny fixed stand-in for the S40 Chroma index (offline reproducibility)
_REG_INDEX = {
    ("GDPR", "breach notification deadline"): {
        "text": "Article 33: notify the supervisory authority without undue delay and, "
                "where feasible, not later than 72 hours after becoming aware of a breach.",
        "ids": [33, 34],
    },
    ("AI Act", "high risk obligations"): {
        "text": "Article 16: providers of high-risk AI systems shall ensure compliance "
                "with the requirements of Section 2 before placing on the market.",
        "ids": [16],
    },
}

def search_regulation(query: str, regulation: str) -> str:
    """Retrieve the most relevant passage from a regulation corpus (GDPR or AI Act).
    Use when the task needs a fact grounded in the law, with article ids to cite."""
    # offline: fuzzy match on keyword; in class this is a Chroma similarity search
    best = None
    for (reg, key), val in _REG_INDEX.items():
        if reg == regulation and any(w in query.lower() for w in key.split()):
            best = val; break
    if best is None:
        return "no relevant passage found"
    return json.dumps({"passage": best["text"], "article_ids": best["ids"]})

SEARCH_REGULATION_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_regulation",
        "description": "Retrieve a grounded passage from GDPR or the EU AI Act. "
                       "Use for any legal obligation, deadline, or article reference. "
                       "Do NOT use for arithmetic.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "regulation": {"type": "string", "enum": ["GDPR", "AI Act"]},
            },
            "required": ["query", "regulation"],
        },
    },
}

print(search_regulation("breach notification deadline", "GDPR"))

{"passage": "Article 33: notify the supervisory authority without undue delay and, where feasible, not later than 72 hours after becoming aware of a breach.", "article_ids": [33, 34]}


### Tool 3: `date_calculator`
A third deterministic tool for reporting deadlines. In the exercise the students build this one.

In [7]:
from datetime import date, timedelta

def date_calculator(start: str, days: int) -> str:
    """Add a number of days to a start date (YYYY-MM-DD). Use for reporting deadlines."""
    y, m, d = map(int, start.split("-"))
    return (date(y, m, d) + timedelta(days=int(days))).isoformat()

DATE_CALCULATOR_SCHEMA = {
    "type": "function",
    "function": {
        "name": "date_calculator",
        "description": "Add days to a start date (YYYY-MM-DD) and return the deadline date. "
                       "Use after finding a deadline length in the regulation.",
        "parameters": {
            "type": "object",
            "properties": {"start": {"type": "string"}, "days": {"type": "integer"}},
            "required": ["start", "days"],
        },
    },
}

# register the two new tools
TOOLS["search_regulation"] = (search_regulation, SEARCH_REGULATION_SCHEMA)
TOOLS["date_calculator"]   = (date_calculator, DATE_CALCULATOR_SCHEMA)
print("Registered tools:", list(TOOLS))
print(date_calculator("2026-10-01", 3))

Registered tools: ['calculator', 'search_regulation', 'date_calculator']
2026-10-04


### Demo B: two tools, one task
A question that needs both retrieval and calculation. We never tell the agent the order;
it infers from the task that it needs the law first, then the date.

In [8]:
result = run_agent(
    "Under GDPR, how many days do I have to notify a breach, and what is the deadline "
    "if the breach happened on 2026-10-01?",
    tool_names=["search_regulation", "date_calculator"],
)
print("\nANSWER:", result["answer"])
print("iterations:", result["iterations"], "| cost: $%.6f" % result["cost_usd"])

# Assert v3.1 on the trajectory
tools_used = [s.get("tool") for s in result["trace"] if "tool" in s]
assert "search_regulation" in tools_used
assert result["iterations"] <= MAX_ITERATIONS
print("trajectory:", tools_used)

[iter 1] tool_call: search_regulation({'query': 'breach notification deadline', 'regulation': 'GDPR'})
          result:    {"passage": "Article 33: notify the supervisory authority without undue delay and, where feasible, not later than 72 hours after becoming aware of a breach.", "article_ids": [33, 34]}
[iter 2] tool_call: date_calculator({'start': '2026-10-01', 'days': 3})
          result:    2026-10-04
[iter 3] final:     Under GDPR Article 33 you must notify within 72 hours; starting 2026-10-01 the deadline is 2026-10-04.

ANSWER: Under GDPR Article 33 you must notify within 72 hours; starting 2026-10-01 the deadline is 2026-10-04.
iterations: 3 | cost: $0.000321
trajectory: ['search_regulation', 'date_calculator']


### The cost of the loop
Every iteration re-sends the whole growing history. Let us measure an agent against a single
direct call on the same task. This is the number behind the module punchline.

In [9]:
# reconstruct the per-call cost of the 3-step agent using the teaching scenario
per_call = [(410, 45), (835, 45), (1260, 120)]   # (input, output) tokens per API call
agent_in  = sum(a for a, _ in per_call)
agent_out = sum(b for _, b in per_call)
agent_cost = agent_in * PRICE_IN + agent_out * PRICE_OUT

single_in, single_out = 410, 120                 # one direct call, no loop
single_cost = single_in * PRICE_IN + single_out * PRICE_OUT

print("agent  : %d in + %d out tokens -> $%.6f" % (agent_in, agent_out, agent_cost))
print("single : %d in + %d out tokens -> $%.6f" % (single_in, single_out, single_cost))
print("ratio  : %.2fx more expensive" % (agent_cost / single_cost))
print("per 1000 tasks: agent $%.2f vs single $%.2f" % (agent_cost*1000, single_cost*1000))

assert round(agent_cost / single_cost, 1) == 4.9
print("\nAutonomy is a budget, not a switch.")

agent  : 2505 in + 210 out tokens -> $0.001644
single : 410 in + 120 out tokens -> $0.000337
ratio  : 4.88x more expensive
per 1000 tasks: agent $1.64 vs single $0.34

Autonomy is a budget, not a switch.


## Transfer to your data (home bonus)
Take your own Module 5 arXiv assistant and wrap its retrieval as a tool, exactly as we wrapped
`search_regulation`. Then run `run_agent` on a two-step question in your project domain and read
the trace. This is Step 1 of the module assignment.

In [10]:
# TODO (home): wrap your arXiv retrieval as a tool and register it in TOOLS
# def arxiv_search(query: str, max_results: int) -> str:
#     """Search arXiv abstracts in your project domain."""
#     ...
# TOOLS["arxiv_search"] = (arxiv_search, ARXIV_SEARCH_SCHEMA)
print("Your agent, your tools. See the assignment brief for the full spec.")

Your agent, your tools. See the assignment brief for the full spec.
